# Kiểm tra DataLoader CIFAR-10 trong notebook
Mỗi bước có `assert`: nếu điều kiện sai, notebook dừng tại bước đó và hiện thông báo.

Notebook đọc dữ liệu CIFAR-10 đã tải tại `data/raw/`, kiểm tra split và lấy thử **một batch của mỗi tập train/validation/test**. Cấu hình kiểm tra là ảnh **224×224**, batch size **32**, `num_workers=0`. Kết quả chỉ xác nhận luồng dữ liệu với cấu hình này; đây chưa phải kiểm tra mọi batch hoặc đo tốc độ.

## 1. Kiểm tra môi trường và thư mục project

`sys.executable` cho biết notebook đang dùng Python nào. Chạy cell này trước để các lệnh import và đường dẫn phía sau hoạt động đúng.

In [1]:
from pathlib import Path
import os
import sys
import json

print('Python đang dùng:', sys.executable)

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in project_candidates
     if (path / 'src' / 'cifar_compare' / 'data.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở notebook từ thư mục project hoặc thư mục notebooks bên trong project.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torchvision

print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA khả dụng:', torch.cuda.is_available())
print('PASS — môi trường và thư mục project đã sẵn sàng.')

Python đang dùng: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\.venv-1\Scripts\python.exe
Project root: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification
PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
CUDA khả dụng: True
PASS — môi trường và thư mục project đã sẵn sàng.


## 2. Chọn cấu hình kiểm tra

Đọc các đường dẫn, seed và kích thước validation từ `configs/default.json`. Ba giá trị bên dưới chỉ áp dụng trong notebook, không sửa file JSON.

- `input_size=224`: mỗi ảnh sau resize có kích thước 224×224.
- `batch_size=32`: gom tối đa 32 ảnh trong một batch.
- `num_workers=0`: nạp dữ liệu trong tiến trình hiện tại, thuận tiện khi học và tìm lỗi. Vẫn có thể training với giá trị 0.

In [2]:
config_path = PROJECT_ROOT / 'configs' / 'default.json'
data_config = json.loads(config_path.read_text(encoding='utf-8-sig'))
data_config.update({
    'input_size': 224,
    'batch_size': 32,
    'num_workers': 0,
})

checks_passed = {}
DATA_CHECKS_PASSED = False
cifar_loaders = None

for key in ('data_root', 'split_path', 'seed', 'val_size',
            'input_size', 'batch_size', 'num_workers'):
    print(f'{key}: {data_config[key]}')

data_root: data/raw
split_path: data/splits/split_indices.json
seed: 42
val_size: 5000
input_size: 224
batch_size: 32
num_workers: 0


## 3. Kiểm tra dữ liệu gốc đã tải

Đây là ảnh **trước resize/normalize**: 50.000 ảnh train và 10.000 ảnh test, mỗi ảnh 32×32×3, kiểu `uint8`, pixel trong 0–255.

Cell dùng `download=False`: nếu dữ liệu thiếu hoặc hỏng, hãy kiểm tra đường dẫn/file trước khi tiếp tục. Nó không bắt đầu tải lại dữ liệu.

In [3]:
from torchvision.datasets import CIFAR10

checks_passed['datasets'] = False
raw_class_names = None
for split_name, is_train, expected_size in (
    ('train gốc', True, 50000),
    ('test gốc', False, 10000),
):
    raw_dataset = CIFAR10(data_config['data_root'], train=is_train, download=False)
    assert len(raw_dataset) == expected_size, f'Sai số ảnh ở {split_name}.'
    assert raw_dataset.data.shape == (expected_size, 32, 32, 3), 'Sai shape ảnh gốc.'
    assert raw_dataset.data.dtype == np.uint8, 'Ảnh gốc phải có dtype uint8.'
    assert 0 <= int(raw_dataset.data.min()) <= int(raw_dataset.data.max()) <= 255
    if raw_class_names is None:
        raw_class_names = list(raw_dataset.classes)
    else:
        assert raw_dataset.classes == raw_class_names, 'Class order của train/test khác nhau.'
    print(split_name, '| shape:', raw_dataset.data.shape, '| dtype:', raw_dataset.data.dtype)
    del raw_dataset

checks_passed['datasets'] = True
print('PASS — CIFAR-10 gốc đầy đủ và đọc được.')

train gốc | shape: (50000, 32, 32, 3) | dtype: uint8
test gốc | shape: (10000, 32, 32, 3) | dtype: uint8
PASS — CIFAR-10 gốc đầy đủ và đọc được.


## 4. Tạo ba DataLoader bằng code của project

Dùng chính `get_dataloaders()` trong `data.py`. Hàm trả về loader train, validation, test và danh sách tên lớp. Nếu chưa có split, hàm sẽ tạo file indices theo cấu hình.

In [4]:
from src.cifar_compare.data import get_dataloaders

checks_passed['loaders'] = False
train_loader, val_loader, test_loader, class_names = get_dataloaders(data_config)
cifar_loaders = (train_loader, val_loader, test_loader)
loaders = {'train': train_loader, 'validation': val_loader, 'test': test_loader}

assert all(loader.num_workers == 0 for loader in cifar_loaders)
assert all(loader.batch_size == data_config['batch_size'] for loader in cifar_loaders)
checks_passed['loaders'] = True
print('PASS — đã tạo train_loader, val_loader và test_loader.')

PASS — đã tạo train_loader, val_loader và test_loader.


## 5. Kiểm tra số mẫu và split

Với `val_size=5000`, cần có **45.000 train / 5.000 validation / 10.000 test**.

Train và validation được tách từ 50.000 ảnh train gốc: indices không được trùng/lặp và hợp lại phải phủ đủ 50.000 ảnh. Tập test là tập riêng do CIFAR-10 cung cấp. File split cần khớp seed và val_size đang dùng.

In [5]:
checks_passed['split'] = False
expected_sizes = {
    'train': 50000 - int(data_config['val_size']),
    'validation': int(data_config['val_size']),
    'test': 10000,
}
for name, loader in loaders.items():
    assert len(loader.dataset) == expected_sizes[name], f'Sai số mẫu {name}.'
    print(f'{name}: {len(loader.dataset):,} ảnh')

train_indices = np.asarray(train_loader.dataset.indices)
val_indices = np.asarray(val_loader.dataset.indices)
assert np.issubdtype(train_indices.dtype, np.integer)
assert np.issubdtype(val_indices.dtype, np.integer)
assert len(np.unique(train_indices)) == len(train_indices), 'Train có indices lặp.'
assert len(np.unique(val_indices)) == len(val_indices), 'Validation có indices lặp.'
overlap = np.intersect1d(train_indices, val_indices)
assert overlap.size == 0, 'Train và validation bị trùng ảnh.'
np.testing.assert_array_equal(
    np.sort(np.concatenate((train_indices, val_indices))),
    np.arange(50000),
    err_msg='Split chưa phủ đủ/chứa indices ngoài 50.000 ảnh train gốc.',
)

saved_split = json.loads(Path(data_config['split_path']).read_text(encoding='utf-8'))
assert saved_split['seed'] == data_config['seed'], 'Seed của split đã lưu khác config; trao đổi với Khang để dùng đúng split.'
assert saved_split['val_size'] == data_config['val_size'], 'val_size của split đã lưu khác config.'
assert saved_split['train_indices'] == train_indices.tolist()
assert saved_split['val_indices'] == val_indices.tolist()
checks_passed['split'] = True
print('Số ảnh trùng train/validation:', overlap.size)
print('PASS — số mẫu và split hợp lệ, file split khớp cấu hình.')

train: 45,000 ảnh
validation: 5,000 ảnh
test: 10,000 ảnh
Số ảnh trùng train/validation: 0
PASS — số mẫu và split hợp lệ, file split khớp cấu hình.


## 6. Kiểm tra tên lớp và phân bố nhãn

**Class order** là thứ tự gán số cho tên lớp, ví dụ `0 = airplane`. Thứ tự này phải giống nhau ở cả ba tập và khớp `class_names` được trả về.

Với cấu hình hiện tại, mỗi lớp cần có 4.500 ảnh train, 500 ảnh validation và 1.000 ảnh test.

In [6]:
checks_passed['classes'] = False
train_base = train_loader.dataset.dataset
val_base = val_loader.dataset.dataset
test_base = test_loader.dataset
assert len(class_names) == 10
assert list(class_names) == raw_class_names
assert train_base.classes == val_base.classes == test_base.classes == class_names
assert train_base.class_to_idx == val_base.class_to_idx == test_base.class_to_idx

train_targets = np.asarray(train_base.targets)[train_indices]
val_targets = np.asarray(val_base.targets)[val_indices]
test_targets = np.asarray(test_base.targets)
counts_by_split = {}
for name, targets in (('train', train_targets), ('validation', val_targets), ('test', test_targets)):
    assert np.issubdtype(targets.dtype, np.integer)
    assert ((targets >= 0) & (targets < 10)).all(), f'Nhãn {name} ngoài 0–9.'
    counts = np.bincount(targets, minlength=10)
    assert int(counts.sum()) == expected_sizes[name]
    assert int(counts.max() - counts.min()) <= 1, f'Phân bố lớp {name} không cân bằng.'
    counts_by_split[name] = counts

print('ID  Class          Train    Val   Test')
for class_id, class_name in enumerate(class_names):
    train_count = counts_by_split['train'][class_id]
    val_count = counts_by_split['validation'][class_id]
    test_count = counts_by_split['test'][class_id]
    print(f'{class_id:>2}  {class_name:<12} {train_count:>6} {val_count:>6} {test_count:>6}')

checks_passed['classes'] = True
print('PASS — class order và phân bố nhãn của cả ba tập đúng.')

ID  Class          Train    Val   Test
 0  airplane       4500    500   1000
 1  automobile     4500    500   1000
 2  bird           4500    500   1000
 3  cat            4500    500   1000
 4  deer           4500    500   1000
 5  dog            4500    500   1000
 6  frog           4500    500   1000
 7  horse          4500    500   1000
 8  ship           4500    500   1000
 9  truck          4500    500   1000
PASS — class order và phân bố nhãn của cả ba tập đúng.


## 7. Lấy và kiểm tra một batch từ từng loader

**Dtype** là kiểu dữ liệu. Ảnh sau transform cần là `torch.float32`; labels cần là `torch.int64`. Batch ảnh có shape `(B, 3, 224, 224)`, nhãn có shape `(B,)`.

Mỗi batch cần có nhãn trong 0–9 và tensor hữu hạn (không NaN/Inf). Pixel sau normalize có thể âm hoặc lớn hơn 1. DataLoader hiện trả tensor CPU; chuyển batch lên GPU là công việc của training loop.

In [7]:
checks_passed['batches'] = False
batch_results = {}
for name, loader in loaders.items():
    images, labels = next(iter(loader))
    expected_batch = min(data_config['batch_size'], len(loader.dataset))
    expected_shape = (expected_batch, 3, data_config['input_size'], data_config['input_size'])
    assert tuple(images.shape) == expected_shape, f'Sai shape batch {name}.'
    assert tuple(labels.shape) == (expected_batch,), f'Sai shape nhãn {name}.'
    assert images.dtype == torch.float32, f'Ảnh {name} phải là float32.'
    assert labels.dtype == torch.int64, f'Nhãn {name} phải là int64.'
    assert torch.isfinite(images).all().item(), f'Ảnh {name} chứa NaN/Inf.'
    assert ((labels >= 0) & (labels < 10)).all().item(), f'Nhãn {name} ngoài 0–9.'
    batch_results[name] = {
        'images_shape': tuple(images.shape),
        'labels_shape': tuple(labels.shape),
        'images_dtype': str(images.dtype),
        'labels_dtype': str(labels.dtype),
    }
    print(f'PASS — batch {name}')
    print('  Ảnh:', tuple(images.shape), images.dtype, '| Nhãn:', tuple(labels.shape), labels.dtype)
    print('  Nhãn min/max:', labels.min().item(), labels.max().item())
    print('  Pixel sau normalize min/max:', round(images.min().item(), 3), round(images.max().item(), 3))
    del images, labels

checks_passed['batches'] = True
print('PASS — đã đọc và kiểm tra batch của cả train, validation và test.')

PASS — batch train
  Ảnh: (32, 3, 224, 224) torch.float32 | Nhãn: (32,) torch.int64
  Nhãn min/max: 0 9
  Pixel sau normalize min/max: -2.118 2.64
PASS — batch validation
  Ảnh: (32, 3, 224, 224) torch.float32 | Nhãn: (32,) torch.int64
  Nhãn min/max: 0 9
  Pixel sau normalize min/max: -2.118 2.64
PASS — batch test
  Ảnh: (32, 3, 224, 224) torch.float32 | Nhãn: (32,) torch.int64
  Nhãn min/max: 0 9
  Pixel sau normalize min/max: -2.118 2.64
PASS — đã đọc và kiểm tra batch của cả train, validation và test.


## 8. Kiểm tra validation/test ổn định và thứ tự lấy mẫu

Train được shuffle để thay đổi thứ tự ảnh khi học. Validation/test cần giữ thứ tự và không bỏ batch cuối. Đọc lại cùng một mẫu validation/test phải cho tensor và label giống nhau, vì các tập này không dùng augmentation ngẫu nhiên.

Cell kiểm tra lại ba mẫu ở mỗi tập và in transforms để bạn đối chiếu.

In [8]:
from torch.utils.data import RandomSampler, SequentialSampler

checks_passed['evaluation_policy'] = False
assert isinstance(train_loader.sampler, RandomSampler), 'Train chưa shuffle.'
for name, loader in (('validation', val_loader), ('test', test_loader)):
    assert isinstance(loader.sampler, SequentialSampler), f'{name} không được shuffle.'
    assert loader.drop_last is False, f'{name} không được bỏ batch cuối.'
    for sample_index in range(min(3, len(loader.dataset))):
        image_a, label_a = loader.dataset[sample_index]
        image_b, label_b = loader.dataset[sample_index]
        assert torch.equal(image_a, image_b), f'Transform {name} không ổn định.'
        assert int(label_a) == int(label_b)
    print(f'PASS — {name}: không shuffle, không bỏ batch cuối, ba mẫu đọc lại giống nhau.')

print('Transform train:', train_base.transform)
print('Transform validation:', val_base.transform)
print('Transform test:', test_base.transform)
checks_passed['evaluation_policy'] = True

PASS — validation: không shuffle, không bỏ batch cuối, ba mẫu đọc lại giống nhau.
PASS — test: không shuffle, không bỏ batch cuối, ba mẫu đọc lại giống nhau.
Transform train: Compose(
    RandomCrop(size=(32, 32), padding=4)
    RandomHorizontalFlip(p=0.5)
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Transform validation: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Transform test: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 9. Đọc kết quả và bước tiếp theo

Nếu mọi cell phía trên chạy qua, cell cuối in `PASS — DataLoader CIFAR-10 đạt kiểm tra cơ bản`.

Bạn đã kiểm chứng dữ liệu/split, class order, dtype và một batch ở cả ba loader với `num_workers=0`. Phúc có thể gửi output cho Khang xác nhận pipeline rồi chuyển sang smoke training trên một tập con nhỏ. `num_workers=4` là cấu hình khác cần kiểm tra riêng nếu nhóm chọn sử dụng; không bắt buộc đổi sang 4 để bắt đầu training.

In [9]:
required_checks = ('datasets', 'loaders', 'split', 'classes', 'batches', 'evaluation_policy')
missing_checks = [name for name in required_checks if not checks_passed.get(name, False)]
assert not missing_checks, f'Chưa hoàn tất các bước: {missing_checks}. Hãy chạy notebook từ trên xuống.'
DATA_CHECKS_PASSED = True
print('PASS — DataLoader CIFAR-10 đạt kiểm tra cơ bản.')
print('Đã kiểm tra: dữ liệu gốc, split, class order, dtype, batch train/validation/test và chính sách validation/test.')
print('Cấu hình:', {key: data_config[key] for key in ('input_size', 'batch_size', 'num_workers', 'seed', 'val_size')})
print('Phạm vi: một batch mỗi loader và ba mẫu validation/test; chưa duyệt mọi batch hoặc kiểm tra worker=4.')

PASS — DataLoader CIFAR-10 đạt kiểm tra cơ bản.
Đã kiểm tra: dữ liệu gốc, split, class order, dtype, batch train/validation/test và chính sách validation/test.
Cấu hình: {'input_size': 224, 'batch_size': 32, 'num_workers': 0, 'seed': 42, 'val_size': 5000}
Phạm vi: một batch mỗi loader và ba mẫu validation/test; chưa duyệt mọi batch hoặc kiểm tra worker=4.


## kiểm tra riêng cấu hình chính thức. Cell này đọc trực tiếp configs/default.json

In [10]:
# Đọc cấu hình chính thức, không ghi đè num_workers thành 0.
official_config = json.loads(
    (PROJECT_ROOT / "configs/default.json").read_text(
        encoding="utf-8-sig"
    )
)

official_train, official_val, official_test, official_classes = (
    get_dataloaders(official_config)
)

print("Cấu hình đang kiểm tra:", {
    key: official_config[key]
    for key in ("input_size", "batch_size", "num_workers")
})

assert official_classes == class_names, "Class order bị khác!"

official_loaders = {
    "train": official_train,
    "validation": official_val,
    "test": official_test,
}

for name, loader in official_loaders.items():
    assert loader.num_workers == official_config["num_workers"]
    assert loader.batch_size == official_config["batch_size"]

    images, labels = next(iter(loader))

    expected_batch = min(
        official_config["batch_size"], len(loader.dataset)
    )
    size = official_config["input_size"]

    assert tuple(images.shape) == (expected_batch, 3, size, size)
    assert tuple(labels.shape) == (expected_batch,)
    assert images.dtype == torch.float32
    assert labels.dtype == torch.int64
    assert torch.isfinite(images).all().item()
    assert ((labels >= 0) & (labels < len(official_classes))).all().item()

    print(
        f"PASS — {name}:",
        tuple(images.shape),
        f"| workers={loader.num_workers}"
    )

print("PASS — cấu hình DataLoader chính thức đọc được cả ba tập.")

Cấu hình đang kiểm tra: {'input_size': 224, 'batch_size': 32, 'num_workers': 4}
PASS — train: (32, 3, 224, 224) | workers=4
PASS — validation: (32, 3, 224, 224) | workers=4
PASS — test: (32, 3, 224, 224) | workers=4
PASS — cấu hình DataLoader chính thức đọc được cả ba tập.
